In [1]:
%cd /content
!rm -rf cdazzdev-mla

/content


In [2]:
!git clone https://github.com/Randimal/cdazzdev-mla.git
%cd cdazzdev-mla
%cd task2_genai
from google.colab import drive
drive.mount('/content/drive')

Cloning into 'cdazzdev-mla'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 107 (delta 43), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 702.74 KiB | 9.76 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/cdazzdev-mla
/content/cdazzdev-mla/task2_genai
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q \
transformers \
datasets \
accelerate \
peft \
trl \
bitsandbytes \
sentencepiece \
huggingface_hub

In [4]:
from huggingface_hub import login, whoami
# login()
whoami()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'type': 'user',
 'id': '653658040917932040d2be0b',
 'name': 'poshitha',
 'fullname': 'Poshitha Malagala',
 'email': 'poshitha.malagala@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1780272000,
 'isPro': False,
 'avatarUrl': '/avatars/d21198f1c6acc93a7fdc01ee7d7a0b45.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'cdazzdev-task2',
   'role': 'write',
   'createdAt': '2026-05-11T12:24:11.012Z'}}}

# Task 2: Lightweight QLoRA Instruction Fine-Tuning

This notebook demonstrates a practical instruction fine-tuning workflow for a Senior Machine Learning Engineer assessment. The goal is not to chase benchmarks. The goal is to show reproducible, explainable, free-tier-compatible GenAI engineering.

We fine-tune a small instruct model with QLoRA, evaluate fixed prompts before and after tuning, save LoRA adapters locally, and optionally upload the adapter to Hugging Face Hub.

## 1. Introduction

QLoRA combines two practical ideas:

- **4-bit quantization**: load the base model with lower-precision weights to reduce GPU memory.
- **LoRA adapters**: train a small number of adapter parameters while keeping the base model frozen.

This makes instruction tuning possible on constrained GPUs such as free Colab runtimes.

## 2. Environment Setup

Use a GPU runtime for training. In Colab, install dependencies if needed. The helper module uses lazy imports, so it can be inspected on CPU, but model loading/training expects GPU-compatible `bitsandbytes`.

In [5]:
# Colab only: uncomment if dependencies are missing.
# !pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

from pathlib import Path
import json
import sys

# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "task2_genai" else Path.cwd()
# TASK2_SRC = PROJECT_ROOT / "task2_genai" / "src"

# if str(TASK2_SRC) not in sys.path:
#     sys.path.insert(0, str(TASK2_SRC))
PROJECT_ROOT = Path("/content/cdazzdev-mla")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd /content/cdazzdev-mla

from task2_genai.src.finetuning_workflow import (
    compare_before_after,
    create_instruction_dataset,
    ensure_task2_directories,
    export_default_training_config,
    load_base_model_and_tokenizer,
    load_instruction_dataset_for_training,
    load_json,
    load_model_with_adapter,
    load_training_config,
    prepare_model_for_lora,
    push_adapter_to_hub,
    resolve_config_paths,
    save_evaluation_results,
    save_sample_prompts,
    train_qlora_adapters,
    generate_response,
)

ensure_task2_directories()
config = resolve_config_paths(export_default_training_config())
config

/content/cdazzdev-mla


{'base_model': 'Qwen/Qwen2.5-3B-Instruct',
 'adapter_output_dir': '/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora',
 'dataset_path': '/content/cdazzdev-mla/task2_genai/data/instruction_dataset.jsonl',
 'evaluation_path': '/content/cdazzdev-mla/task2_genai/evaluation/before_vs_after_results.json',
 'sample_prompts_path': '/content/cdazzdev-mla/task2_genai/data/sample_prompts.json',
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 4,
 'learning_rate': 0.0002,
 'warmup_steps': 5,
 'logging_steps': 5,
 'save_strategy': 'epoch',
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'load_in_4bit': True,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_compute_dtype': 'bfloat16',
 'bnb_4bit_use_double_quant': True,
 'seed': 42,
 'max_new_tokens': 128}

## 3. Dataset Preparation

The dataset uses a simple instruction/response format. A small high-quality dataset is appropriate here because the goal is to teach response style and format, not broad new knowledge.

Each row has:

- `instruction`: user request
- `response`: desired assistant answer

During preprocessing, rows are converted to the model's chat template with `tokenizer.apply_chat_template()`.

In [6]:
dataset_rows = create_instruction_dataset(Path(config["dataset_path"]))
sample_prompts = save_sample_prompts(Path(config["sample_prompts_path"]))

print(f"Training examples: {len(dataset_rows)}")
print(f"Evaluation prompts: {len(sample_prompts)}")
dataset_rows[:3]

Training examples: 12
Evaluation prompts: 4


[{'instruction': 'Explain QLoRA to a junior machine learning engineer.',
  'response': 'QLoRA fine-tunes a quantized language model using small LoRA adapter weights. The base model is loaded in 4-bit precision to reduce GPU memory, while only the adapter matrices are trained. This makes instruction tuning practical on limited hardware.'},
 {'instruction': 'Summarize why structured outputs improve AI workflows.',
  'response': 'Structured outputs make model responses easier to validate, log, test, and pass between components. Instead of parsing free-form text, downstream code can rely on predictable fields and handle missing or invalid values safely.'},
 {'instruction': 'Give three production risks in an LLM application.',
  'response': 'Three production risks are malformed model outputs, hidden prompt or data drift, and external API failures. Good systems mitigate these with validation, observability, fallbacks, and monitoring.'}]

## 4. Baseline Model Evaluation

Before training, generate baseline outputs on fixed prompts. Keeping prompts fixed makes the before/after comparison fair and reproducible.

In [7]:
# This cell downloads the base model and should be run on a GPU runtime.
base_model, tokenizer = load_base_model_and_tokenizer(config)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [8]:
print(base_model.device)
!nvidia-smi

cuda:0
Mon May 11 14:55:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |    5409MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------------

In [9]:
baseline_responses = []
for prompt in sample_prompts:
    response = generate_response(
        base_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    baseline_responses.append(response)
    print("PROMPT:", prompt)
    print("BASELINE:", response)
    print("-" * 80)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT: Explain QLoRA in simple engineering terms.
BASELINE: QLoRA (Quantized LoRA) is a technique used to make large models like transformers more manageable and faster to run, especially on devices with limited resources. Let's break it down in simple engineering terms:

### What is LoRA?
LoRA stands for Lightweight Recursive Adapter. It's a method that allows you to add small, trainable parameters to a large model. These small parameters help the model learn some of the important features without significantly increasing its size.

### What is QLoRA?
QLoRA is a quantized version of LoRA. Instead of using floating-point numbers (which take up more memory), QLoRA
--------------------------------------------------------------------------------
PROMPT: Why are structured outputs useful in production LLM systems?
BASELINE: Structured outputs are particularly useful in production Large Language Models (LLMs) for several reasons:

1. **Clarity and Consistency**: Structured outputs provide 

## 5. QLoRA Fine-Tuning

The model is prepared for k-bit training, then LoRA adapters are attached. Only adapter parameters are trained.

Important tradeoffs:

- Lower memory than full fine-tuning
- Smaller artifacts because adapters are saved separately
- Less flexible than updating all model weights
- Can overfit quickly on tiny datasets, so keep epochs conservative

In [10]:
train_dataset = load_instruction_dataset_for_training(
    config["dataset_path"],
    tokenizer,
)
train_dataset[0]["text"][:500]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nExplain QLoRA to a junior machine learning engineer.<|im_end|>\n<|im_start|>assistant\nQLoRA fine-tunes a quantized language model using small LoRA adapter weights. The base model is loaded in 4-bit precision to reduce GPU memory, while only the adapter matrices are trained. This makes instruction tuning practical on limited hardware.<|im_end|>\n'

In [11]:
qlora_model = prepare_model_for_lora(base_model, config)
qlora_model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [12]:
trainer = train_qlora_adapters(
    qlora_model,
    tokenizer,
    train_dataset,
    config,
)

print("Saved adapter to:", config["adapter_output_dir"])

Adding EOS to train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,4.179337


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Saved adapter to: /content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora


## 6. Post-Training Evaluation

Now generate responses from the adapter model using the same prompts. The comparison is qualitative because this dataset is intentionally small.

In [13]:
# If you restarted the notebook after training, use:
# tuned_model, tokenizer = load_model_with_adapter(config)

tuned_model = qlora_model
fine_tuned_responses = []
for prompt in sample_prompts:
    response = generate_response(
        tuned_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    fine_tuned_responses.append(response)
    print("PROMPT:", prompt)
    print("FINE-TUNED:", response)
    print("-" * 80)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


PROMPT: Explain QLoRA in simple engineering terms.
FINE-TUNED: QL
--------------------------------------------------------------------------------
PROMPT: Why are structured outputs useful in production LLM systems?
FINE-TUNED: Structured
--------------------------------------------------------------------------------
PROMPT: List common mistakes in notebook-based fine-tuning.
FINE-TUNED: Here a
--------------------------------------------------------------------------------
PROMPT: Compare LoRA adapters with full fine-tuning.
FINE-TUNED: Lo
--------------------------------------------------------------------------------


## 7. Sample Generations

The structured comparison below is saved under `task2_genai/evaluation/` for reproducibility.

In [14]:
comparison_rows = compare_before_after(
    sample_prompts,
    baseline_responses,
    fine_tuned_responses,
)

evaluation_path = save_evaluation_results(
    comparison_rows,
    Path(config["evaluation_path"]),
)
print("Saved evaluation to:", evaluation_path)
comparison_rows

Saved evaluation to: /content/cdazzdev-mla/task2_genai/evaluation/before_vs_after_results.json


[{'prompt': 'Explain QLoRA in simple engineering terms.',
  'baseline_response': "QLoRA (Quantized LoRA) is a technique used to make large models like transformers more manageable and faster to run, especially on devices with limited resources. Let's break it down in simple engineering terms:\n\n### What is LoRA?\nLoRA stands for Lightweight Recursive Adapter. It's a method that allows you to add small, trainable parameters to a large model. These small parameters help the model learn some of the important features without significantly increasing its size.\n\n### What is QLoRA?\nQLoRA is a quantized version of LoRA. Instead of using floating-point numbers (which take up more memory), QLoRA",
  'fine_tuned_response': 'QL',
  'notes': 'Review clarity, task focus, formatting, and whether the fine-tuned response better matches the dataset style.'},
 {'prompt': 'Why are structured outputs useful in production LLM systems?',
  'baseline_response': "Structured outputs are particularly useful

## 8. Saving Adapters

LoRA adapters are saved separately from the base model. This is practical because the adapter is much smaller than the full model and can be loaded on top of the original base model later.

In [15]:
adapter_dir = Path(config["adapter_output_dir"])
print("Adapter directory:", adapter_dir)
print("Files:", list(adapter_dir.glob("*")) if adapter_dir.exists() else "Adapter not found yet")

Adapter directory: /content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora
Files: [PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/adapter_config.json'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/tokenizer.json'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/chat_template.jinja'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/checkpoint-6'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/README.md'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/checkpoint-3'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/tokenizer_config.json'), PosixPath('/content/cdazzdev-mla/task2_genai/outputs/qwen2_5_3b_task2_lora/adapter_model.safetensors')]


## 9. Optional Hugging Face Upload

Upload only if you want to publish or back up the adapter. Authenticate first with `huggingface_hub.notebook_login()`.

In [16]:
# Optional upload. Uncomment and set your repo id.
from huggingface_hub import notebook_login
notebook_login()
push_adapter_to_hub(tuned_model, tokenizer, repo_id="poshitha/task2-qwen-qlora-adapter", private=False)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 61.1kB / 59.9MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mppe4z495e/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


## 10. Engineering Reflection

### Why QLoRA was chosen

QLoRA is suitable for free-tier GPUs because 4-bit quantization reduces memory usage while LoRA limits the number of trainable parameters.

### Why PEFT is practical

PEFT trains adapters instead of all model weights. This reduces GPU memory, training time, storage cost, and upload size.

### Free-tier GPU limitations

Free GPUs have limited VRAM, session timeouts, variable availability, and slower training. The config uses batch size 1, gradient accumulation, one epoch, and short sequence length to stay practical.

### Tradeoffs vs full fine-tuning

Full fine-tuning can adapt more deeply but requires much more compute and storage. LoRA is cheaper and easier to manage, but it may be less expressive for large behavior changes.

### Why small high-quality datasets can work

For style and format adaptation, clear examples often matter more than volume. A tiny dataset will not teach broad knowledge, but it can teach the expected response pattern.

### Common mistakes

- Training without baseline outputs
- Changing evaluation prompts after tuning
- Using too many epochs on a tiny dataset
- Forgetting to save the tokenizer with adapters
- Treating qualitative improvement as benchmark proof
- Running 4-bit training on CPU-only environments

### Hugging Face integration

The adapter can be pushed to the Hub after authentication. Users load the base model and then attach the adapter with PEFT.